# Matching

Conceptually, Matching pairs treated units with untreated units to construct a
synthetic control group. This quasi-experimental method mitigates selection bias
by balancing the dataset and estimating the Average Treatment Effect on the
Treated (ATT) on the *matched* data.

There are two primary steps:
1. **Estimation & Matching**: Fit a model to predict $P(D_i = 1 | X_i)$, generate
   predicted probabilities ($\hat{p}_i$) for the entire dataset, and pair treated
   units with untreated units based on the similarity of these scores.
2. **Analysis**: Use the matched dataset from Step 1 to estimate the ATT.

We will use the [Lalonde dataset](
    https://vincentarelbundock.github.io/Rdatasets/doc/MatchIt/lalonde.html
) for this task. The treatment variable, `treat`, is a binary indicator of
whether an individual was assigned to a subsidized job training program. The
goal is to estimate the treatment's effect on `re78` (real earnings in 1978). By
balancing the sample on pre-treatment characteristics—such as age, education,
and prior earnings (`re74`, `re75`)—we will replicate the original experimental
benchmarks using Propensity Score Matching (PSM).

## Setting things up

Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.stats import ttest_rel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
import statsmodels.api as sm

Read dataset

In [ ]:
URL = 'https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/MatchIt/lalonde.csv'
df = pd.read_csv(URL).drop(columns=['rownames'])
print('No. observations:', len(df))
display(df.head())
display(df.isna().sum())

### Feature Engineering

As always, we'll do a little bit of feature engineering prior to fitting our models.

In [ ]:
# Add constant
df['const'] = 1

# Encode `race`
df = pd.concat([df, pd.get_dummies(df[['race']], dtype=int)], axis=1)

# Covariates we will use to predict probability of being treated
psm_cols_X = [
    'const', 'age', 'educ', 'married', 'nodegree', 're74', 're75', 'race_black',
    'race_hispan'
]

# Covariates used to estimate ATE
cols_X = ['treat'] + psm_cols_X

## Naive ATE Estimation
Let's estimate the ATE with a simple Linear Regression model.


In [ ]:
m0 = sm.OLS(endog=df['re78'], exog=df[cols_X]).fit(cov_type='HC1')
print(m0.summary())  # +$1.5K

## Propensity Score Matching

### 1. Fitting the Propensity model

We will use a Logistic Regression to learn what factors influence $D_i = 1$.
Do note we could use a more complex Machine Learning classifier for this task!

First, we will do a quick layer of feature engineering.

In [ ]:
# Init and fit model
psm = LogisticRegression(penalty=None, max_iter=100000, random_state=42)
psm.fit(X=df[psm_cols_X], y=df['treat'])

# Make predictions
psm_pred = psm.predict(df[psm_cols_X])
psm_prob = psm.predict_proba(df[psm_cols_X])[:, 1]

Evaluate classifier

In [ ]:
print(f'F1 Score: {f1_score(df['treat'], psm_pred):.2f}')
print(f'ROC AUC Score: {roc_auc_score(df['treat'], psm_prob):.2f}')
display(confusion_matrix(df['treat'], psm_pred) / len(df))  # FPR is kinda high

The classifier we just trained isn't perfect. In fact, its False Positive Ratio is
somewhat high. However, since we'll match observations using the predicted
probabilities, it's the ROC AUC that we care about the most (because it plots the TPR
and FPR for *all* cutoff values).


In [ ]:
# Assign predicted probability column
df['p_hat'] = psm_prob

# Save probabilities in numpy arrays
y0 = df.loc[df['treat'].eq(0), ['p_hat']].values
y1 = df.loc[df['treat'].eq(1), ['p_hat']].values

### Balance dataset
We are going to calculate the euclidean distance in the predicted probabilties between
all treated and untreated units, and then match each treated unit to its closest
neighbor (with replacement).

In [ ]:
cdist(y0, y1).shape[1]

In [ ]:
# Distance matrix
dist = cdist(y0, y1, metric='euclidean')  # Rows: Control and Columns: Treatment

# Maximum distance allowed
THR = 0.05

# For loop to find nearest-neighbor
for i in range(dist.shape[1]):

    # Extract distances for i-th treated unit
    a = dist[:, i]

    # Get index of nearest neighbor
    idx = np.argmin(a)
    val = a[idx]

    # Match if couple meets condition
    if val <= THR:
        idx_1 = df[df['treat'].eq(1)].iloc[i].name.item()
        idx_0 = df[df['treat'].eq(0)].iloc[idx].name.item()
        df.loc[idx_1, 'match'] = idx_0

In [ ]:
df.head()

In [ ]:
example = df.sample(frac=1, random_state=42).head()
example

In [ ]:
example.iloc[4].name.item()

Declare matched table

In [ ]:
df_balanced = pd.concat(
    [
        df.loc[df['match'].notna()],
        df.loc[df.loc[df['match'].notna(), 'match']]
    ],
    axis=0,
    ignore_index=True
)

### Estimate ATE using balanced dataset

In [ ]:
m1 = sm.OLS(
    endog=df_balanced['re78'],
    exog=df_balanced[cols_X]
).fit(cov_type='HC1')
print(m1.summary())

And since we have paired observations, we can actually use a paired
*t*-test as a robustness check.

In [ ]:
# Save ordered incomes
re78_0 = df.loc[df.loc[df['match'].notna(), 'match'], 're78'].values  # Control
re78_1 = df.loc[df['match'].notna(), 're78'].values  # Treatment

# Unscaled ATE
print(f'Estimated ATE: ${np.mean(re78_1 - re78_0):.2f}')

# Perform test
ttest = ttest_rel(re78_0, re78_1)
print(f'p-value: {ttest.pvalue:.3f}')

## Big Brain Time
Before matching, we found that the training increased `re_78` by $1.5K. After matching,
our estimate increased to $1.9K. Why do you think this happened?